# COSMoS BC, NCIt Layer Discriminator Probe

Test whether the nine instrument question-container BCs sit under NCIt super-class `C211913` ('CDISC QRS Instruments Questions') by ancestry.

**Background.** The previous version of this probe used the NCI EVS REST API to fetch concept properties including `Contributing_Source` (P322). EVS REST has been unresponsive (60 to 150 seconds per concept fetch). This rewrite drops the API dependency and uses the cached `Thesaurus.txt` FLAT file already downloaded by the test-codes track.

**Trade-off.** FLAT carries `parents` but not `P322`. So this probe answers the C211913 ancestry discriminator only. The complementary P322 ('CDISC contributing source') discriminator is deferred until either (a) EVS REST is responsive, or (b) a property-specific extract is identified on the NCI FTP, or (c) the OWL file is parsed.

**What this probe does establish.** If every question-container BC has C211913 in its NCIt ancestor chain, then NCIt's own structural classification confirms these are container concepts under the CDISC QRS Instruments Questions branch, distinct from the instrument identities themselves.


In [ ]:
import pandas as pd
from pathlib import Path

# Directory structure - notebook is in notebooks/ subfolder
BASE_DIR    = Path.cwd().parent
INTERIM_DIR = BASE_DIR / 'interim'
INTERIM_FILE = INTERIM_DIR / 'COSMoS_BC_DSS.xlsx'

# Cross-track reference to test-codes downloads (cached by SDTM_CT_NCIt_Enrich)
NCIT_FLAT_FILE = BASE_DIR.parent / 'sdtm-test-codes' / 'downloads' / 'Thesaurus.txt'

assert INTERIM_FILE.exists(), f'Interim file not found: {INTERIM_FILE}'
assert NCIT_FLAT_FILE.exists(), (
    f'NCIt FLAT file not found: {NCIT_FLAT_FILE}\n'
    f'Run SDTM_CT_NCIt_Enrich.ipynb first to download and cache it.'
)

print(f'BC_DSS:    {INTERIM_FILE}')
print(f'NCIt FLAT: {NCIT_FLAT_FILE}')

## Resolve question-container BCs to NCIt codes


In [ ]:
df = pd.read_excel(INTERIM_FILE, sheet_name='BC_DSS')
bc = df.drop_duplicates('BC_Name')[['BC_Name', 'NCIt_Code']].copy()

question_containers = [
    '6MWT Functional Test Question',
    'ADAS-Cog CDISC Version Functional Test Question',
    'CES Questionnaire Question',
    'EQ-5D-5L Questionnaire Question',
    'AIMS Clinical Classification Question',
    'APACHE II Clinical Classification Question',
    'HAMA Questionnaire Question',
    'KPS Scale Questionnaire Question',
    'Tanner Scale-Boy Clinical Classification Question',
]

qc = bc[bc['BC_Name'].isin(question_containers)][['BC_Name', 'NCIt_Code']].reset_index(drop=True)
missing = set(question_containers) - set(qc['BC_Name'])
if missing:
    print(f'WARNING: not found in BC_DSS: {sorted(missing)}')
print(qc.to_string())

## Load NCIt FLAT

Same column layout as `SDTM_CT_NCIt_Enrich.ipynb`. The full file is loaded; no row filtering at read time because the parent lookup needs the entire concept graph for transitive walks.


In [ ]:
FLAT_COLS = ['code', 'url', 'parents', 'synonyms', 'definition',
             'display_name', 'concept_status', 'semantic_type', 'subset_membership']

print(f'Reading {NCIT_FLAT_FILE.name} ...')
flat = pd.read_csv(
    NCIT_FLAT_FILE,
    sep='\t',
    header=None,
    names=FLAT_COLS,
    dtype=str,
    na_filter=False,
)
print(f'Loaded {len(flat):,} NCIt concepts')

## Build parent and label lookups

`parents` is pipe-separated NCIt codes. Each concept can have multiple direct parents.


In [ ]:
parent_map = {}
label_map  = {}
for _, r in flat.iterrows():
    code_ = r['code']
    parents_str = r['parents']
    # Prefer display_name; fall back to first synonym (preferred term per FLAT convention)
    label = r['display_name']
    if not label and r['synonyms']:
        label = r['synonyms'].split('|')[0]
    label_map[code_] = label
    if parents_str:
        parent_map[code_] = set(p for p in parents_str.split('|') if p)
    else:
        parent_map[code_] = set()

print(f'Parent map: {len(parent_map):,} concepts')
print(f'Label map:  {len(label_map):,} concepts')

## Transitive ancestor walk


In [ ]:
def ancestors(code_, parent_map, max_depth=50):
    """Return the set of all ancestor codes reachable from code_, BFS up."""
    seen = set()
    frontier = parent_map.get(code_, set())
    depth = 0
    while frontier and depth < max_depth:
        new_frontier = set()
        for c in frontier:
            if c in seen:
                continue
            seen.add(c)
            new_frontier |= parent_map.get(c, set())
        frontier = new_frontier - seen
        depth += 1
    return seen

## Probe: does C211913 appear in each container's ancestor chain?


In [ ]:
TARGET = 'C211913'

results = []
for _, r in qc.iterrows():
    code_ = r['NCIt_Code']
    if code_ not in parent_map:
        results.append({
            'BC_Name':  r['BC_Name'],
            'NCIt_Code': code_,
            'In_FLAT':   False,
            'Under_C211913': None,
            'Direct_Parents': '(not in FLAT)',
        })
        continue
    anc = ancestors(code_, parent_map)
    direct = sorted(parent_map.get(code_, set()))
    direct_lbl = '; '.join(f'{c} ({label_map.get(c, "?")})' for c in direct)
    results.append({
        'BC_Name':  r['BC_Name'],
        'NCIt_Code': code_,
        'In_FLAT':   True,
        'Under_C211913': TARGET in anc,
        'Direct_Parents': direct_lbl,
    })

results_df = pd.DataFrame(results)
print(results_df[['BC_Name', 'NCIt_Code', 'In_FLAT', 'Under_C211913']].to_string())
print()
print('Direct parents per container:')
for _, r in results_df.iterrows():
    print(f'  {r["BC_Name"]}')
    print(f'    {r["Direct_Parents"]}')

## C211913 children: NCIt vs COSMoS materialization

Reverse the parent relation to find every NCIt concept whose direct parent is C211913. Compare against the set of NCIt codes present in COSMoS BC_DSS to compute the materialization gap.


In [ ]:
c211913_children = sorted(
    code_ for code_, parents in parent_map.items() if TARGET in parents
)
print(f'NCIt direct children of {TARGET}: {len(c211913_children)}')

cosmos_codes = set(bc['NCIt_Code'].dropna())
materialized = [c for c in c211913_children if c in cosmos_codes]
print(f'Materialized as COSMoS BCs:        {len(materialized)}')
print(f'Not in COSMoS:                     {len(c211913_children) - len(materialized)}')

if materialized:
    mat_df = pd.DataFrame([
        {'NCIt_Code': c,
         'NCIt_Label': label_map.get(c, '?'),
         'BC_Name': bc[bc['NCIt_Code'] == c]['BC_Name'].iloc[0] if (bc['NCIt_Code'] == c).any() else ''}
        for c in materialized
    ])
    print()
    print(mat_df.to_string())
    identical = (mat_df['NCIt_Label'] == mat_df['BC_Name']).all()
    print(f'\nAll {len(mat_df)} BC names match NCIt label exactly: {identical}')

## Summary

Two facts established without any API calls:

1. **C211913 ancestry of the nine question-container BCs.** See the `Under_C211913` column above.
2. **Materialization ratio of C211913 children.** Direct NCIt children vs COSMoS BCs that carry those codes.

Both come from the cached `Thesaurus.txt` FLAT file. No network access required at probe time.

Note: in the current data, C211913 is the direct parent of every tested container, not merely a distant ancestor. This is a sharper structural claim than the transitive walk strictly requires.

**Deferred.** The P322 Contributing_Source check, which would test the complementary claim that instrument identities (e.g. '6 Minute Walk Functional Test') carry CDISC as a contributing source. P322 is not in FLAT. Pending either an EVS REST recovery, an FTP property extract, or OWL parsing.
